<a href="https://colab.research.google.com/github/Fares-pr0g/ML-journey-ep-2-Experimenting-with-NN-s-in-PyTorch/blob/main/Learning_PyTorch_08%3A%20Character_level_Language_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Character-level Language Modeling Using LSTM's

## DATASET: Let's work on the famous book The 48 Laws Of Power by Robert Greene

In [28]:
from google.colab import files

uploaded= files.upload()

Saving The 48 Laws Of Power.pdf to The 48 Laws Of Power (1).pdf


## Data Preprocessing:

In [29]:
!pip install pypdf

In [30]:
from pypdf import PdfReader

# turning the pdf book to a .txt file
pdf_path="/content/The 48 Laws Of Power.pdf"
reader= PdfReader(pdf_path)

text= ""

for page in reader.pages:
  text+= page.extract_text() + "\n"

In [31]:
print(len(text))
print(text[:2000])

1380195

THE 48 LAWS OF POWER

ROBERT GREENE has a degree in classical studies
and has been an editor at Esquire and oflmt magazines.
He is also a playwright and lives in Los Angeles.
JOOST ELFERS is the producer of 772: 48 Laws ofPawer
and also of
The Sam: Language cyFB:fi-tkdays
with Gary Goldschneider
The Semi Languages ofReiatiansiuyu
with Gary Goldschneider
Play with Hmr Facet‘
with Saxton Freymann

P
O
W
E
R
ROBERT GREENE
A JOOST ELFFERS PRODUCTION
P
PROFILE BOOKS

This paperback edition published in 2000
Reprinted 200}, 2002
First published in Great Britain in 1998 by
Profile Books Ltd
58A Hatnon Garden
London ECIN RLX
First published in the United States in 1998 by
Viking, a division of Penguin Putnam Inc.
Copyright ® Robert Greene aridjoost Elflers, 1998
A portion of this work first appeared in '17w Uzne Reader
Typeset in BE Baskerville
Printed and bound in Italy by
Legoprint S.p.a.
—
Lavis (TN)
The moral right of the authors has been asserted.
All rights reserved. Without lim

In [32]:
with open("48_laws_of_power.txt", "w") as f:
  f.write(text)

In [33]:
import numpy as np
# Reading and processing the text
# We don't need the acknowledgement section in for our data

with open("48_laws_of_power.txt", "r") as f:
  text= f.read()
end_indx= text.find(" As Man said, “When we\nfight you, we make sure you can’t get away.”\n")
text=text[:end_indx]
char_set= set(text)

print('Total Length:', len(text))
print("Unique Characters:", len(char_set))

Total Length: 1315513
Unique Characters: 110


In [34]:
# Encoding the text with a chat2int method
import numpy as np

chars_sorted= sorted(char_set)
char2int= {ch:i for i,ch in enumerate(chars_sorted)}
char_array= np.array(chars_sorted)
text_encoded= np.array([char2int[ch] for ch in text], dtype= np.int32)

# Let's run a small test
print('Text encoded shape:', text_encoded.shape)
print(text[:21], '== Encoding ==>', text_encoded[:21])
print(text_encoded[23:36], '== Decoding ==>', text[23:36])

Text encoded shape: (1315513,)

THE 48 LAWS OF POWER == Encoding ==> [ 0 53 41 38  1 21 25  1 45 34 56 52  1 48 39  1 49 48 56 38 51]
[51 48 35 38 51 53  1 40 51 38 38 47 38] == Decoding ==> ROBERT GREENE


### Device Agnostic Code:

In [35]:
import torch

device= "cuda" if torch.cuda.is_available() else 'cpu'
device

'cuda'

### Set the dataset architecture:

In [36]:
import torch
from torch.utils.data import Dataset

seq_length= 40
chunk_size=seq_length+1
text_chunks= [text_encoded[i:i+chunk_size]
              for i in range(len(text_encoded)-chunk_size)]

class TextDataset(Dataset):

  def __init__(self, text_chunks):
    self.text_chunks= text_chunks

  def __len__(self):
    return len(self.text_chunks)

  def __getitem__(self, idx):
    text_chunk = self.text_chunks[idx]
    return text_chunk[:-1].long().to(device), text_chunk[1:].long().to(device)

seq_dataset= TextDataset(torch.tensor(text_chunks))


In [37]:
# Let's create a dataloader

from torch.utils.data import DataLoader

BATCH_SIZE= 64
torch.manual_seed(42)
seq_dl= DataLoader(seq_dataset, BATCH_SIZE, shuffle= True)


In [38]:
# test
from torch import nn

data_sample = next(iter(seq_dl))
print(data_sample)

embedding= nn.Embedding(len(char_array),16).to(device)

embedding(data_sample[0]).shape

[tensor([[ 1, 72, 69,  ..., 83, 82,  1],
        [67,  1, 57,  ..., 84, 82, 84],
        [77, 83, 84,  ..., 64,  1, 76],
        ...,
        [75, 88, 13,  ..., 68, 81,  1],
        [82, 83, 68,  ..., 68,  1, 64],
        [72, 77, 64,  ..., 76, 68, 15]], device='cuda:0'), tensor([[ 72,  69,   1,  ...,  82,   1,  83],
        [  1,  57, 104,  ...,  82,  84,  64],
        [ 83,  84,  64,  ...,   1,  76,  64],
        ...,
        [ 88,  13,   1,  ...,  81,   1,  64],
        [ 83,  68,  67,  ...,   1,  64,  75],
        [ 77,  64,  75,  ...,  68,  15,   0]], device='cuda:0')]


torch.Size([64, 40, 16])

## Model 0: Unidirectional LSTM

In [39]:
import torch.nn as nn

class RNN (nn.Module):
  def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
    super().__init__()
    self.embedding= nn.Embedding(vocab_size, embed_dim)
    self.rnn_hidden_size= rnn_hidden_size
    self.rnn= nn.LSTM(embed_dim, rnn_hidden_size, batch_first= True)
    self.fc= nn.Linear(rnn_hidden_size, vocab_size)

  def forward(self, X, hidden, cell):
    out=self.embedding(X).unsqueeze(1)
    out, (hidden, cell) = self.rnn(out, (hidden, cell))
    out= self.fc(out).reshape(out.size(0), -1)
    return out, hidden, cell

  def init_hidden(self, batch_size):
    hidden= torch.zeros(1, batch_size, self.rnn_hidden_size)
    cell= torch.zeros(1, batch_size, self.rnn_hidden_size)
    return hidden.to(device), cell.to(device)


VOCAB_SIZE= len(char_array)
EMBED_DIM= 256
RNN_HIDDEN_SIZE= 512
torch.manual_seed(42)
model_0= RNN(VOCAB_SIZE, EMBED_DIM, RNN_HIDDEN_SIZE).to(device)
model_0


RNN(
  (embedding): Embedding(110, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=110, bias=True)
)

In [41]:
loss_fn= nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(model_0.parameters(), lr= 0.001)

NUM_EPOCHS = 10000
torch.manual_seed(42)

from timeit import default_timer as timer
start_time= timer()

for epoch in range(NUM_EPOCHS):
  hidden, cell = model_0.init_hidden(BATCH_SIZE)
  seq_batch, target_batch = next(iter(seq_dl))
  seq_batch, target_batch = seq_batch.to(device), target_batch.to(device)
  optimizer.zero_grad()
  loss = 0
  for c in range(seq_length):
    pred, hidden, cell = model_0(seq_batch[:, c], hidden, cell)
    loss += loss_fn(pred, target_batch[:, c])
  loss.backward()
  optimizer.step()
  loss = loss.item()/seq_length
  if epoch % 500 == 0:
    print(f'Epoch {epoch} loss: {loss:.4f}')

end_time= timer()
print(f'Total training time: {end_time-start_time:.2f} seconds')

Epoch 0 loss: 1.8135
Epoch 500 loss: 1.6631
Epoch 1000 loss: 1.5606
Epoch 1500 loss: 1.6223
Epoch 2000 loss: 1.5238
Epoch 2500 loss: 1.4742
Epoch 3000 loss: 1.5628
Epoch 3500 loss: 1.4082
Epoch 4000 loss: 1.4295
Epoch 4500 loss: 1.5277
Epoch 5000 loss: 1.5656
Epoch 5500 loss: 1.4648
Epoch 6000 loss: 1.4237
Epoch 6500 loss: 1.3958
Epoch 7000 loss: 1.4564
Epoch 7500 loss: 1.3258
Epoch 8000 loss: 1.3540
Epoch 8500 loss: 1.3886
Epoch 9000 loss: 1.3183
Epoch 9500 loss: 1.4169
Total training time: 1494.95 seconds


### Now, let's create the function that generates text using our model!

In [44]:
from torch.distributions.categorical import Categorical

def sample(model, starting_str, len_generated_text=500, scale_factor=1.0):

  encoded_input= torch.tensor([char2int[s] for s in starting_str]).to(device)

  encoded_input= torch.reshape(encoded_input, (1, -1))
  generated_str= starting_str

  model.eval()
  hidden, cell= model.init_hidden(1)
  for c in range(len(starting_str)-1):
    _,hidden, cell= model(encoded_input[:,c].view(1), hidden, cell)

  last_char= encoded_input[:,-1]
  for i in range(len_generated_text):
    logits, hidden, cell= model(last_char.view(1), hidden, cell)
    logits= torch.squeeze(logits, 0)
    scaled_logits= logits * scale_factor
    m= Categorical(logits=scaled_logits)
    last_char= m.sample()
    generated_str += str(char_array[last_char])

  return generated_str

In [53]:
torch.manual_seed(42)
generated_text= sample(model_0, starting_str="politics")

import time



for char in generated_text:
    print(char, end="", flush=True)
    time.sleep(0.05)

politics;
then have although he drawn as the end, feelings you seek, and everything soldier, Lustig, however, having work without
somehowing the Mess, starts the belease has the promotion remained: He never re-
veen waspers‘, more wardly, enormous in seduction. And kept themselves diplomaty. It was surfered the island of the most deal of Austria, Towards, lack of
violences, but since sure that seduce
themselves.
In eval, Duveen’s suckers?
Once the same disrectly seemed like a
large old and the abandone 

**-> This looks promising, but let's try a lower temperature**

**Terminology Alert:** The scaling factor, 𝛼 ,
can be interpreted as an analog to the temperature in physics. Higher temperatures result in more
entropy or randomness versus more predictable behavior at lower temperatures. By scaling the logits
with 𝛼<1, the probabilities computed by the softmax function become more uniform.

In [54]:
torch.manual_seed(42)
generated_text= sample(model_0, starting_str="politics", scale_factor=2.0)

import time



for char in generated_text:
    print(char, end="", flush=True)
    time.sleep(0.05)

politics starts to death. All of the game is nothing that seem to see the streets. He lost the same thing to see the work of the Roman suitor,
which is the problem when the powerful people can get the one that has its catch and explaining the people had a patron had the most country that he did not surferent all the surface of the palace of the powerful than to stir up and under the post of the end of the court.
Interpretation
Louis XIV had no power in the problem. And indeed even though the same temple